# Forecast VAR: World Cup 2026 Prediction Agent

This notebook is the end-to-end story for Episode 2. It uses deterministic mock mode so it can run without API cost.

## 1. Validate the tournament field

The first rule of a prediction agent: know who actually qualified. Italy is intentionally checked as absent.

In [ ]:
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()/"src"))
from forecast_var.tools import validate_tournament_field, get_source_registry, forecast_group, rank_teams
from forecast_var.runner import run_agent
from forecast_var.backtest import run_backtest
validation = validate_tournament_field()
validation

## 2. Source registry

The agent is designed to support many source types, but the repo bundles only legal/reproducible demo inputs.

In [ ]:
registry = get_source_registry()
[(s["id"], s["status"], s["category"]) for s in registry["sources"]]

## 3. Group D forecast

This is a viewer-friendly example because it includes the co-host USA and Australia.

In [ ]:
group_d = forecast_group("D", sims=3000)
group_d["group_probabilities"]

In [ ]:
from IPython.display import Image, display
import subprocess, os
env = {**os.environ, "PYTHONPATH": "src"}
subprocess.run(["python", "scripts/generate_figures.py"], check=True, env=env)
display(Image(filename="figures/group_d_winner_probabilities.png"))

## 4. Top tournament favourites

This is a demo title-probability proxy, not a full bracket simulation.

In [ ]:
top = rank_teams(10)
top["ranked"][:10]

In [ ]:
display(Image(filename="figures/top_tournament_favourites.png"))

## 5. Scenario analysis

A useful prediction agent should handle scenarios without pretending they are confirmed news.

In [ ]:
scenario_answer = run_agent("If France loses 80 rating points because of injuries, how does that change the top favourites?", mode="grounded_mock")
scenario_answer.model_dump()

## 6. Baseline vs grounded agent evaluation

The baseline makes unsupported claims. The grounded agent must use tools, skills, citations, probabilities, and abstention.

In [ ]:
import subprocess, json, os
env = {**os.environ, "PYTHONPATH": "src"}
subprocess.run(["python", "scripts/evaluate.py", "--mode", "baseline_mock"], check=True, env=env)
subprocess.run(["python", "scripts/evaluate.py", "--mode", "grounded_mock"], check=True, env=env)
subprocess.run(["python", "scripts/generate_figures.py"], check=True, env=env)
base = json.load(open("reports/summary_baseline_mock.json"))
grounded = json.load(open("reports/summary_grounded_mock.json"))
base, grounded

In [ ]:
display(Image(filename="figures/eval_summary.png"))

## 7. Forecast model backtest harness

A prediction agent needs both answer evaluation and model evaluation. This toy backtest computes log loss and Brier score on a small historical sample.

In [ ]:
backtest = run_backtest()
{k:v for k,v in backtest.items() if k != "rows"}

## 8. Live OpenAI API path

The notebook uses mock mode for reproducibility. To run the real OpenAI agent with MCP tools:

```bash
export OPENAI_API_KEY="your_key_here"
PYTHONPATH=src python scripts/run_agent.py "Who is favourite to win Group D?" --mode openai --model gpt-4.1-mini
```

The live implementation is in `src/forecast_var/openai_agent.py`.